**PP434 portfolio** · Jakob Kilian Bilan · CC5, scraper

# Scraping Wikipedia's table of tax rates

Source: https://en.wikipedia.org/wiki/List_of_countries_by_tax_rates . Output: `cc5_tax_rates_tidy.csv`, tidy (long) format: one row per jurisdiction and tax.

In [ ]:
import re
import pandas as pd

URL = "https://en.wikipedia.org/wiki/List_of_countries_by_tax_rates"
# Wikipedia blocks requests from code that does not identify itself (HTTP 403), so send a User-Agent
HEADERS = {"User-Agent": "PP434 course notebook (LSE student project)"}
OUTPUTS = ["cc5_tax_rates_tidy.csv"]

## 1. Scrape

In [ ]:
def get_table(source):
    # match= keeps only the table whose text contains 'Inheritance' (the main rates table)
    tables = pd.read_html(source, match="Inheritance", storage_options=HEADERS) if str(source).startswith("http") \
        else pd.read_html(source, match="Inheritance")
    return tables[0]

raw = get_table(URL)
print(raw.shape)
raw.head()

## 2. Clean and normalise

Rules (documented so the cleaning can be checked): footnote markers like `[12]` are removed; the headline rate is the first percentage in a cell (`19–25%` gives 25, `31,4%` gives 31.4); a cell starting with *No* counts as 0%; for VAT the rate marked *(standard)* is used if there is one, and values above 32% are treated as special-goods rates, not standard rates; flat-tax rows that only fill *Lowest* use that rate as the top rate; wealth tax is kept as a category (Yes, No, Inactive, Not listed, Other).

In [ ]:
def flatten_columns(df):
    # the table has two header rows (e.g. 'Individual income' over 'Lowest' / 'Highest'): join them
    cols = []
    for col in df.columns:
        parts = [str(p) for p in (col if isinstance(col, tuple) else (col,))]
        parts = [p for i, p in enumerate(parts) if p not in parts[:i] and not p.startswith("Unnamed")]
        cols.append(" ".join(parts).strip())
    df.columns = cols
    return df

def find_col(df, *keywords):
    for c in df.columns:
        if all(k.lower() in c.lower() for k in keywords):
            return c
    raise KeyError(f"No column containing {keywords}; columns are {list(df.columns)}")

PCT = re.compile(r"(\d+(?:[.,]\d+)?)\s*%")

def first_pct(text):
    """Headline rate: the first percentage in the cell ('19–25%' -> 25, '31,4%' -> 31.4)."""
    if pd.isna(text):
        return None
    text = re.sub(r"\[[^\]]*\]", "", str(text))          # drop footnote markers like [12]
    m = PCT.search(text)
    if m:
        return float(m.group(1).replace(",", "."))
    if re.match(r"\s*no\b", text, flags=re.I):            # 'No', 'No, but ...' -> no such tax
        return 0.0
    return None

def standard_vat(text):
    """VAT/GST/sales: prefer the rate labelled '(standard', else the first percentage."""
    if pd.isna(text):
        return None
    text = re.sub(r"\[[^\]]*\]", "", str(text))
    m = re.search(r"(\d+(?:[.,]\d+)?)\s*%\s*\(standard", text, flags=re.I)
    value = float(m.group(1).replace(",", ".")) if m else first_pct(text)
    return value if value is None or value <= 32 else None   # >32% is a special-goods rate (e.g. 300% on alcohol), not a standard rate

def wealth_status(text):
    if pd.isna(text) or str(text).strip() in ("", "—", "-"):
        return "Not listed"
    t = str(text).strip().lower()
    for key in ("yes", "no", "inactive"):
        if t.startswith(key):
            return key.capitalize()
    return "Other"

In [ ]:
def tidy_table(raw):
    df = flatten_columns(raw.copy())
    c_country = df.columns[0]
    c_corp = find_col(df, "corporate")
    c_low = find_col(df, "individual", "lowest")
    c_high = find_col(df, "individual", "highest")
    c_cg = find_col(df, "capital")
    c_wealth = find_col(df, "wealth")
    c_inh = find_col(df, "inherit")
    c_vat = find_col(df, "vat")
    out = pd.DataFrame({"jurisdiction": df[c_country].astype(str).str.replace(r"\[.*?\]", "", regex=True).str.strip()})
    out["wealth_tax"] = df[c_wealth].map(wealth_status)
    out["corporate"] = df[c_corp].map(first_pct)
    high, low = df[c_high].map(first_pct), df[c_low].map(first_pct)
    out["income_top"] = high.where(high.notna(), low)     # flat-tax rows only fill 'Lowest'
    out["capital_gains"] = df[c_cg].map(first_pct)
    out["inheritance"] = df[c_inh].map(first_pct)
    out["vat"] = df[c_vat].map(standard_vat)
    out = out.drop_duplicates("jurisdiction")
    tidy = out.melt(id_vars=["jurisdiction", "wealth_tax"], var_name="tax", value_name="top_rate_pct")
    return tidy.dropna(subset=["top_rate_pct"]).sort_values(["jurisdiction", "tax"]).reset_index(drop=True)

tidy = tidy_table(raw)
print(tidy.shape)
tidy.head(10)

## 3. Checks

In [ ]:
wide = tidy.pivot_table(index=["jurisdiction", "wealth_tax"], columns="tax", values="top_rate_pct").reset_index()
print("jurisdictions:", tidy["jurisdiction"].nunique())
print(tidy.groupby("tax")["top_rate_pct"].describe().round(1))
print(wide["wealth_tax"].value_counts())
# Spot-check a few rows against the web page by eye
wide[wide["jurisdiction"].isin(["United Kingdom", "United States", "France", "Germany", "Norway", "Spain"])]

In [ ]:
assert tidy["top_rate_pct"].between(0, 100).all()
assert {"income_top", "vat", "corporate"} <= set(tidy["tax"])
tidy.to_csv(OUTPUTS[0], index=False)
print("saved", OUTPUTS[0], len(tidy), "rows")

In [ ]:
# Colab only: download the output to your computer, then upload it to the data folder of your GitHub repository
try:
    from google.colab import files
    for f in OUTPUTS:
        files.download(f)
except ImportError:
    print("Not running in Colab: the files are in the working folder:", OUTPUTS)